# Multi-label Classification of Antimicrobial Peptide (AMP) Mechanisms

**Preparatory project for the DDLS PhD position: *Towards a mechanism-aware multi-label prediction framework for antimicrobial peptides***

This notebook demonstrates a progressively more sophisticated pipeline for multi-label prediction of AMP activity, directly motivated by the core challenge: *how do you build a reliable predictive framework when the training labels themselves are systematically wrong?*

### What this notebook covers
1. Data loading and multi-label annotation parsing (DRAMP database)
2. Feature engineering from peptide sequences
3. Label distribution analysis and co-occurrence visualisation
4. Classical multi-label baselines: Binary Relevance and Classifier Chains
5. Deep learning: PyTorch multi-label neural network
6. Label noise handling: Positive-Unlabelled (PU) learning
7. Placeholder architecture for multi-modal data fusion
8. Discussion: limitations, label noise, and the path to mechanism-aware modelling

## 1. Setup and Imports

In [ ]:
# Uncomment to install in Kaggle/Colab
# !pip install scikit-multilearn torch -q

import pandas as pd
import numpy as np
import re
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, accuracy_score, f1_score

from skmultilearn.problem_transform import ClassifierChain

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load AMP Dataset (DRAMP)

In [ ]:
url = "https://raw.githubusercontent.com/Aridoge13/AMP-datasets/main/DRAMP_2024_clean.csv"

try:
    df = pd.read_csv(url)
    print(f"Loaded DRAMP dataset: {len(df)} entries")
except Exception as e:
    print(f"Remote load failed ({e}). Using synthetic fallback.")
    # Expanded synthetic dataset (100 peptides) for more meaningful model evaluation
    rng = np.random.default_rng(42)
    aa_alphabet = 'ACDEFGHIKLMNPQRSTVWY'
    activities = ['antibacterial', 'antibiofilm', 'antifungal', 'antiviral', 'immunomodulatory']

    def random_peptide(length):
        return ''.join(rng.choice(list(aa_alphabet), length))

    def random_activity():
        # antibacterial is highly prevalent (reflects real DRAMP bias)
        selected = ['antibacterial']
        for act in activities[1:]:
            if rng.random() < 0.3:
                selected.append(act)
        return ', '.join(selected)

    synthetic_data = {
        'Peptide_sequence': [random_peptide(rng.integers(8, 40)) for _ in range(100)],
        'Activity': [random_activity() for _ in range(100)]
    }
    df = pd.DataFrame(synthetic_data)
    print(f"Synthetic dataset: {len(df)} peptides, {len(activities)} activity classes")

df.head()

## 3. Multi-label Annotation Parsing

In [ ]:
def parse_activity(text):
    labels = [l.strip().lower() for l in str(text).split(',') if pd.notna(text)]
    return list(set([l for l in labels if l]))

df['label_list'] = df['Activity'].apply(parse_activity)

unique_labels = sorted(set([label for sublist in df['label_list'] for label in sublist]))
print(f"Unique activity labels ({len(unique_labels)}): {unique_labels}")

mlb = MultiLabelBinarizer(classes=unique_labels)
y = mlb.fit_transform(df['label_list'])
label_names = mlb.classes_
print(f"Label matrix shape: {y.shape}")

# Label frequency
label_counts = y.sum(axis=0)
print("\nLabel frequencies:")
for name, count in sorted(zip(label_names, label_counts), key=lambda x: -x[1]):
    print(f"  {name}: {count} ({100*count/len(df):.1f}%)")

## 4. Label Distribution Visualisation

Two key visualisations before modelling: label frequency and label co-occurrence.
These are not merely diagnostic — they reveal the structural biases in the annotation space
that any mechanism-aware model must account for.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel 1: Label frequency bar chart ---
sorted_idx = np.argsort(label_counts)[::-1]
sorted_names = [label_names[i] for i in sorted_idx]
sorted_counts = label_counts[sorted_idx]

bars = axes[0].bar(sorted_names, sorted_counts, color='steelblue', edgecolor='white')
axes[0].set_title('Label Frequency (DRAMP)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Activity label')
axes[0].tick_params(axis='x', rotation=30)
for bar, count in zip(bars, sorted_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(int(count)), ha='center', va='bottom', fontsize=9)

# --- Panel 2: Label co-occurrence heatmap ---
cooccurrence = y.T @ y  # shape: (n_labels, n_labels)
# Normalise by diagonal (conditional probability P(j | i))
diag = np.diag(cooccurrence).astype(float)
diag[diag == 0] = 1  # avoid division by zero
cooccurrence_norm = cooccurrence / diag[:, None]

sns.heatmap(
    cooccurrence_norm,
    xticklabels=label_names,
    yticklabels=label_names,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=axes[1],
    vmin=0, vmax=1
)
axes[1].set_title('Label Co-occurrence (P(col | row))', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=40)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('label_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nNote: antibacterial dominates heavily — a known annotation bias in DRAMP.")
print("Co-occurrence structure is the basis for Classifier Chains and label embedding approaches.")

## 5. Feature Engineering from Peptide Sequences

In [ ]:
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_list = list(amino_acids)

def aa_composition(seq):
    seq = seq.upper()
    length = len(seq)
    return [seq.count(aa)/length if length > 0 else 0 for aa in aa_list]

def dipeptide_composition(seq):
    seq = seq.upper()
    dipeps = [a+b for a in aa_list for b in aa_list]
    count = Counter([seq[i:i+2] for i in range(len(seq)-1)])
    total = max(len(seq)-1, 1)
    return [count.get(dp, 0)/total for dp in dipeps]

def charge_at_pH7(seq):
    seq = seq.upper()
    return seq.count('K') + seq.count('R') - seq.count('D') - seq.count('E')

def gravy(seq):
    hydropathy = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,
                  'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,
                  'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
    total = sum(hydropathy.get(aa, 0) for aa in seq.upper())
    return total / len(seq) if seq else 0

print("Extracting features...")
X_aa  = np.array([aa_composition(s) for s in df['Peptide_sequence']])
X_dp  = np.array([dipeptide_composition(s) for s in df['Peptide_sequence']])
X_ext = np.array([[charge_at_pH7(s), gravy(s), len(s)] for s in df['Peptide_sequence']])

X = np.hstack([X_aa, X_dp, X_ext])
print(f"Feature matrix: {X.shape}  (samples x features)")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 6. Classical Multi-label Baselines

Binary Relevance and Classifier Chains serve as interpretable baselines.
BR treats labels as independent; CC explicitly models label dependencies
by chaining classifiers in sequence — more biologically appropriate given
the strong co-occurrence structure observed above.

In [ ]:
def multi_label_metrics(y_true, y_pred, label_names, model_name):
    y_pred_dense = np.array(y_pred.todense()) if hasattr(y_pred, 'todense') else y_pred
    metrics = {
        'Hamming loss':     hamming_loss(y_true, y_pred_dense),
        'Exact match':      accuracy_score(y_true, y_pred_dense),
        'F1 micro':         f1_score(y_true, y_pred_dense, average='micro', zero_division=0),
        'F1 macro':         f1_score(y_true, y_pred_dense, average='macro', zero_division=0),
    }
    per_label = f1_score(y_true, y_pred_dense, average=None, zero_division=0)
    print(f"\n=== {model_name} ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics, per_label

# Binary Relevance
br = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
br.fit(X_train, y_train)
y_pred_br = br.predict(X_test)
br_metrics, br_per_label = multi_label_metrics(y_test, y_pred_br, label_names, 'Binary Relevance (RF)')

# Classifier Chains
cc = ClassifierChain(RandomForestClassifier(n_estimators=100, random_state=42))
cc.fit(X_train, y_train)
y_pred_cc = cc.predict(X_test)
cc_metrics, cc_per_label = multi_label_metrics(y_test, y_pred_cc, label_names, 'Classifier Chains (RF)')

In [ ]:
# Per-label F1 comparison
x = np.arange(len(label_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, br_per_label, width, label='Binary Relevance', color='steelblue')
ax.bar(x + width/2, cc_per_label, width, label='Classifier Chains', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-label F1: Binary Relevance vs Classifier Chains', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend()
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
plt.tight_layout()
plt.savefig('per_label_f1_baselines.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nNote: low F1 on minority labels reflects class imbalance — not just model weakness.")

## 7. Deep Learning: PyTorch Multi-label Neural Network

A feedforward network with sigmoid output — one unit per label, trained with
binary cross-entropy loss. This serves as a neural baseline and a stepping stone
toward the representation learning approaches (graph NNs, transformers, embeddings)
central to the PhD project.

In [ ]:
class MultiLabelMLP(nn.Module):
    """
    Feedforward network for multi-label classification.
    Architecture: input -> [FC -> BN -> ReLU -> Dropout] x 3 -> sigmoid output
    """
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        # Note: no sigmoid here — BCEWithLogitsLoss applies it internally (more numerically stable)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def predict(self, x, threshold=0.5):
        logits = self.forward(x)
        return (torch.sigmoid(logits) >= threshold).float()


def train_mlp(X_tr, y_tr, X_val, y_val, epochs=60, batch_size=32, lr=1e-3):
    input_dim  = X_tr.shape[1]
    output_dim = y_tr.shape[1]

    model = MultiLabelMLP(
        input_dim=input_dim,
        hidden_dims=[512, 256, 128],
        output_dim=output_dim,
        dropout=0.3
    ).to(device)

    # Class-weighted loss: upweight minority labels
    pos_weight = torch.tensor(
        (y_tr.shape[0] - y_tr.sum(0)) / (y_tr.sum(0) + 1e-6),
        dtype=torch.float32
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    X_tr_t  = torch.FloatTensor(X_tr).to(device)
    y_tr_t  = torch.FloatTensor(y_tr).to(device)
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.FloatTensor(y_val).to(device)

    dataset  = TensorDataset(X_tr_t, y_tr_t)
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()
        train_losses.append(epoch_loss / len(loader))
        val_losses.append(val_loss)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs} | train loss: {train_losses[-1]:.4f} | val loss: {val_losses[-1]:.4f}")

    return model, train_losses, val_losses


print("Training PyTorch MLP...")
mlp_model, train_losses, val_losses = train_mlp(X_train, y_train, X_test, y_test)

In [ ]:
# Training curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train loss', color='steelblue')
ax.plot(val_losses,   label='Val loss',   color='coral')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.set_title('MLP Training Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('mlp_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# MLP evaluation
mlp_model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test).to(device)
    y_pred_mlp = mlp_model.predict(X_test_t).cpu().numpy()

mlp_metrics, mlp_per_label = multi_label_metrics(y_test, y_pred_mlp, label_names, 'MLP (PyTorch)')

In [ ]:
# Three-way per-label F1 comparison
x = np.arange(len(label_names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, br_per_label,  width, label='Binary Relevance', color='steelblue')
ax.bar(x,         cc_per_label,  width, label='Classifier Chains', color='coral')
ax.bar(x + width, mlp_per_label, width, label='MLP (PyTorch)',    color='mediumseagreen')
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-label F1: Baseline Comparison', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend()
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
plt.tight_layout()
plt.savefig('per_label_f1_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Label Noise Handling: Positive-Unlabelled (PU) Learning

Standard supervised learning assumes labels are correct. In AMP databases this is
systematically false: a peptide annotated only as 'antibacterial' may genuinely possess
antifungal or immunomodulatory activity that was simply never tested.

This is the **Positive-Unlabelled (PU) learning** problem: we observe **confirmed positives**
and **unlabelled** instances — not true negatives. Treating unlabelled examples as negatives
biases the decision boundary and depresses recall for minority labels.

**Approach implemented here:** Two-step PU learning per label
1. Step 1 — Identify reliable negatives using a spy-based approach
2. Step 2 — Train a standard classifier on positives + reliable negatives

This is applied independently per label (consistent with Binary Relevance).

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone


class PULearner(BaseEstimator, ClassifierMixin):
    """
    Two-step PU learner for a single binary label.

    Step 1: Mix a random spy fraction of positives into the unlabelled set.
            Train a classifier; score all unlabelled examples.
            Use spy scores to choose a threshold: unlabelled examples scoring
            below the minimum spy score are 'reliable negatives' (RN).

    Step 2: Train a final classifier on P + RN.

    Reference: Liu et al. (2002), "Partially Supervised Classification of Text Documents".
    """

    def __init__(self, base_estimator=None, spy_rate=0.15, random_state=42):
        self.base_estimator = base_estimator or RandomForestClassifier(n_estimators=100, random_state=42)
        self.spy_rate = spy_rate
        self.random_state = random_state

    def fit(self, X, y):
        """
        Parameters
        ----------
        X : array (n_samples, n_features)
        y : array (n_samples,)  — binary, 1 = confirmed positive, 0 = unlabelled
        """
        rng = np.random.default_rng(self.random_state)
        pos_idx = np.where(y == 1)[0]
        unl_idx = np.where(y == 0)[0]

        if len(pos_idx) == 0 or len(unl_idx) == 0:
            # Degenerate case: fall back to standard fit
            self.classifier_ = clone(self.base_estimator).fit(X, y)
            self.reliable_neg_threshold_ = None
            return self

        # --- Step 1: spy injection ---
        n_spies = max(1, int(len(pos_idx) * self.spy_rate))
        spy_idx = rng.choice(pos_idx, size=n_spies, replace=False)
        non_spy_pos_idx = np.setdiff1d(pos_idx, spy_idx)

        X_step1 = np.vstack([X[non_spy_pos_idx], X[unl_idx], X[spy_idx]])
        y_step1 = np.concatenate([
            np.ones(len(non_spy_pos_idx)),
            np.zeros(len(unl_idx)),     # unlabelled treated as negative in step 1
            np.zeros(len(spy_idx))      # spies also treated as negative
        ])

        clf1 = clone(self.base_estimator).fit(X_step1, y_step1)

        # Score spies to calibrate threshold
        spy_scores = clf1.predict_proba(X[spy_idx])[:, 1]
        self.reliable_neg_threshold_ = spy_scores.min()

        # Reliable negatives: unlabelled examples scoring below spy minimum
        unl_scores = clf1.predict_proba(X[unl_idx])[:, 1]
        reliable_neg_mask = unl_scores < self.reliable_neg_threshold_
        reliable_neg_idx = unl_idx[reliable_neg_mask]

        n_rn = len(reliable_neg_idx)
        if n_rn == 0:
            # No reliable negatives found — lower threshold to bottom quartile
            threshold_q25 = np.percentile(unl_scores, 25)
            reliable_neg_idx = unl_idx[unl_scores <= threshold_q25]

        # --- Step 2: train on P + RN ---
        X_step2 = np.vstack([X[pos_idx], X[reliable_neg_idx]])
        y_step2 = np.concatenate([
            np.ones(len(pos_idx)),
            np.zeros(len(reliable_neg_idx))
        ])

        self.classifier_ = clone(self.base_estimator).fit(X_step2, y_step2)
        self.n_reliable_neg_ = len(reliable_neg_idx)
        return self

    def predict_proba(self, X):
        return self.classifier_.predict_proba(X)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


class MultiLabelPUClassifier(BaseEstimator, ClassifierMixin):
    """
    Binary Relevance wrapper applying PU learning independently per label.
    """

    def __init__(self, spy_rate=0.15, random_state=42):
        self.spy_rate = spy_rate
        self.random_state = random_state

    def fit(self, X, Y):
        self.learners_ = []
        for j in range(Y.shape[1]):
            pu = PULearner(spy_rate=self.spy_rate, random_state=self.random_state)
            pu.fit(X, Y[:, j])
            self.learners_.append(pu)
        return self

    def predict(self, X, threshold=0.5):
        preds = np.column_stack([l.predict(X, threshold) for l in self.learners_])
        return preds


print("Training PU learner (per-label)...")
pu_clf = MultiLabelPUClassifier(spy_rate=0.15, random_state=42)
pu_clf.fit(X_train, y_train)
y_pred_pu = pu_clf.predict(X_test)
pu_metrics, pu_per_label = multi_label_metrics(y_test, y_pred_pu, label_names, 'PU Learning (RF)')

print("\nReliable negatives identified per label:")
for name, learner in zip(label_names, pu_clf.learners_):
    rn = getattr(learner, 'n_reliable_neg_', 'N/A')
    thr = getattr(learner, 'reliable_neg_threshold_', None)
    thr_str = f"{thr:.3f}" if thr is not None else 'N/A'
    print(f"  {name}: {rn} reliable negatives (spy threshold: {thr_str})")

In [ ]:
# Full model comparison
x = np.arange(len(label_names))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 1.5*width, br_per_label,  width, label='Binary Relevance', color='steelblue')
ax.bar(x - 0.5*width, cc_per_label,  width, label='Classifier Chains', color='coral')
ax.bar(x + 0.5*width, mlp_per_label, width, label='MLP (PyTorch)',    color='mediumseagreen')
ax.bar(x + 1.5*width, pu_per_label,  width, label='PU Learning',      color='mediumpurple')
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-label F1: All Models', fontweight='bold')
ax.set_ylim(0, 1.2)
ax.legend(loc='upper right')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
plt.tight_layout()
plt.savefig('per_label_f1_all_models_pu.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
summary = pd.DataFrame({
    'Metric': list(br_metrics.keys()),
    'Binary Relevance': list(br_metrics.values()),
    'Classifier Chains': list(cc_metrics.values()),
    'MLP (PyTorch)': list(mlp_metrics.values()),
    'PU Learning': list(pu_metrics.values()),
}).set_index('Metric').round(4)
print("\n", summary.to_string())

## 9. Towards Multi-modal Data Fusion

The PhD project's key departure from sequence-only modelling is the integration of
**multi-omics, microscopy, and spectroscopy** data to capture mechanistic signatures
that are invisible at the sequence level.

This section sketches the proposed fusion architecture. The actual multi-modal datasets
are held by the Wenzel lab and not publicly available; this is a design prototype.

### Conceptual data modalities

| Modality | Example features | Dimensionality |
|---|---|---|
| Sequence | AA/dipeptide composition, physicochemical | ~423 |
| Transcriptomics | Differential gene expression under AMP treatment | ~4,000 |
| Proteomics | Protein abundance changes | ~1,000 |
| Morphology (microscopy) | Cell shape, membrane integrity features | ~50–200 |
| Spectroscopy | FTIR/Raman spectral signatures | ~500–2,000 |

### Fusion strategy options
- **Early fusion**: concatenate all modalities before the model — simple but ignores modality-specific structure
- **Late fusion**: train one model per modality, combine predictions — loses cross-modal interactions
- **Intermediate (cross-modal attention)**: encode each modality separately, then use cross-attention to integrate — captures interactions while preserving modality-specific representations

In [ ]:
class ModalityEncoder(nn.Module):
    """Encodes a single data modality into a shared embedding space."""
    def __init__(self, input_dim, embed_dim, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x):
        return self.encoder(x)


class CrossModalAttentionFusion(nn.Module):
    """
    Fuses multiple modality embeddings via multi-head cross-attention.

    Each modality embedding attends to all others, then a final MLP
    maps the fused representation to multi-label predictions.

    Note: This is a design prototype. In the PhD, this architecture
    would be applied to real multi-modal AMP data from the Wenzel lab.
    """

    def __init__(self, modality_dims, embed_dim, n_labels, n_heads=4, dropout=0.2):
        """
        Parameters
        ----------
        modality_dims : list[int]  — input dimension of each modality
        embed_dim     : int        — shared embedding dimension (must be divisible by n_heads)
        n_labels      : int        — number of output labels
        n_heads       : int        — attention heads
        """
        super().__init__()
        self.encoders = nn.ModuleList([
            ModalityEncoder(d, embed_dim, dropout) for d in modality_dims
        ])
        self.n_modalities = len(modality_dims)

        # Cross-modal attention: each modality as query, others as key/value
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(embed_dim)

        # Prediction head
        fused_dim = embed_dim * self.n_modalities
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_labels)
        )

    def forward(self, modality_inputs):
        """
        Parameters
        ----------
        modality_inputs : list[Tensor]  — one tensor per modality, shape (batch, dim_i)

        Returns
        -------
        logits : Tensor (batch, n_labels)
        """
        embeddings = [enc(x) for enc, x in zip(self.encoders, modality_inputs)]
        # Stack: (batch, n_modalities, embed_dim)
        stacked = torch.stack(embeddings, dim=1)

        # Cross-attention: each modality attends to all others
        attended, _ = self.cross_attn(stacked, stacked, stacked)
        attended = self.layer_norm(stacked + attended)  # residual connection

        # Flatten attended embeddings
        fused = attended.flatten(start_dim=1)  # (batch, n_modalities * embed_dim)
        return self.classifier(fused)

    def predict(self, modality_inputs, threshold=0.5):
        logits = self.forward(modality_inputs)
        return (torch.sigmoid(logits) >= threshold).float()


# Prototype instantiation (synthetic modality dimensions)
modality_dims = {
    'sequence':       X.shape[1],   # actual sequence features (this notebook)
    'transcriptomics': 4000,         # placeholder — differential expression
    'proteomics':      1000,         # placeholder — protein abundances
    'morphology':       128,         # placeholder — microscopy features
    'spectroscopy':    1024,         # placeholder — FTIR/Raman
}

fusion_model = CrossModalAttentionFusion(
    modality_dims=list(modality_dims.values()),
    embed_dim=128,
    n_labels=y.shape[1],
    n_heads=4
).to(device)

n_params = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print("CrossModalAttentionFusion — architecture summary:")
print(fusion_model)
print(f"\nTotal trainable parameters: {n_params:,}")
print("\nNote: This model cannot be trained here — the transcriptomics, proteomics,")
print("morphology, and spectroscopy modalities are not publicly available.")
print("The architecture is designed for the curated Wenzel-lab multi-modal datasets.")

## 10. Discussion: Label Noise, Mechanism Awareness, and the Path Forward

### The core problem

The DRAMP database, like most biological annotation resources, carries **systematic label noise**:

- **Under-labelling**: Peptides annotated only as 'antibacterial' may have antifungal or
  immunomodulatory activities that were never tested. The label 0 does not mean 'inactive'
  — it means 'untested'. This is precisely the PU learning setting.
- **Ontological inconsistency**: Different studies use incompatible terminology
  (e.g., 'antifungal' vs 'anti-Candida'), introducing label ambiguity that cannot
  be resolved without ontology-informed curation.
- **Historical oversimplification**: For decades, AMPs were assumed to act exclusively
  via membrane disruption. This has systematically biased the annotation space away
  from metabolic interference, ribosomal inhibition, and immune modulation.

### What this notebook demonstrates

| Component | Relevance to PhD |
|---|---|
| Binary Relevance / Classifier Chains | Classical baselines; CC captures label co-occurrence |
| Label co-occurrence heatmap | Reveals structural dependencies the model must learn |
| PyTorch MLP with class weighting | Neural baseline; handles label imbalance explicitly |
| PU learning | Addresses the core annotation problem: 0 ≠ negative |
| CrossModalAttentionFusion | Architecture for multi-modal integration of Wenzel-lab data |

### What the PhD will extend

- Replace sequence features with ontology-informed, multi-modal representations
- Move from spy-based PU to probabilistic label models (e.g., noise transition matrices)
- Implement graph neural networks over mechanism ontologies
- Systematic benchmarking on the three curated Wenzel-lab datasets
- Biological validation: do high-confidence novel predictions hold up experimentally?

The central claim is that mechanism-aware label construction — not just better models —
is the prerequisite for reliable AMP prediction. This notebook is a first demonstration
of that thesis.

In [ ]:
print("Notebook complete. Figures saved:")
print("  label_analysis.png            — label frequency + co-occurrence heatmap")
print("  per_label_f1_baselines.png    — BR vs CC comparison")
print("  mlp_training_curve.png        — PyTorch MLP learning curve")
print("  per_label_f1_all_models_pu.png — full four-model comparison")